# Silver Processing — Order Items

Cleans `ecommerce.bronze.bronze_order_items` (183,378 rows, 92 daily files) into `ecommerce.silver.silver_order_items`:

* **String columns** (`customer_id`, `unit_price_currency`, `channel`, `coupon_code`) — trim spaces and strip trailing special characters (raw data is already mostly clean; pass-through safe)
* **`quantity`** — integer: the word form `Two` (33,138 rows) maps to `2`; digits `1`/`3`/`4` cast as-is. Verified: no other word forms exist in the data
* **`unit_price`** — strip `$` prefixes and commas, cast to `double` (all values are whole numbers, range 0–114,991)
* **`discount_pct`** — stored as a fraction: `10%` → `0.1` (strip `%`, divide by 100)
* **Types**: `dt` date, `order_ts` timestamp, `order_id`/`item_seq` int, `product_id` long, `tax_amount` int
* **`coupon_code`** nulls (119,548 rows with no coupon) are preserved — null is a valid "no coupon"
* Metadata columns (`source_file`, `ingested_at`) pass through untouched
* `order_ts` and `dt` verified consistent on every bronze row — no repair needed

In [0]:
from pyspark.sql.functions import col, trim, upper, when, lit, regexp_replace

bronze_table = "ecommerce.bronze.bronze_order_items"
target_table = "ecommerce.silver.silver_order_items"

df_bronze_order_items = spark.table(bronze_table)

# Clean a string column: trim spaces and strip trailing special characters
def clean(column_name):
    return regexp_replace(trim(col(column_name)), "[^A-Za-z0-9]+$", "")

df_silver_order_items = (
    df_bronze_order_items
    # String columns: trim + strip trailing special characters
    .withColumn("customer_id", clean("customer_id"))
    .withColumn("unit_price_currency", clean("unit_price_currency"))
    .withColumn("channel", clean("channel"))
    .withColumn("coupon_code", clean("coupon_code"))
    # quantity: word form 'Two' -> 2, digits cast as-is (only word form in the data)
    .withColumn(
        "quantity",
        when(upper(trim(col("quantity"))) == "TWO", lit(2))
        .otherwise(trim(col("quantity")).cast("int")),
    )
    # unit_price: strip $ prefix and thousands commas -> double
    .withColumn(
        "unit_price",
        regexp_replace(trim(col("unit_price")), "[$,]", "").cast("double"),
    )
    # discount_pct: '10%' -> 0.1 (strip %, divide by 100)
    .withColumn(
        "discount_pct",
        (regexp_replace(trim(col("discount_pct")), "%", "").cast("double") / 100),
    )
    # Types already correct in bronze for: dt (date), order_ts (timestamp),
    # order_id/item_seq (int), product_id (long), tax_amount (int)
)

# Idempotent full load into the silver table
(
    df_silver_order_items.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
# Verify the silver table
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT_IF(quantity IS NULL) AS null_qty,
      MIN(quantity) AS min_qty,
      MAX(quantity) AS max_qty,
      COUNT_IF(quantity = 2) AS qty_2_rows
    FROM ecommerce.silver.silver_order_items
""").show()

spark.sql("""
    SELECT
      MIN(unit_price) AS min_price,
      MAX(unit_price) AS max_price,
      COUNT_IF(unit_price IS NULL) AS null_price,
      MIN(discount_pct) AS min_disc,
      MAX(discount_pct) AS max_disc,
      COUNT_IF(discount_pct IS NULL) AS null_disc
    FROM ecommerce.silver.silver_order_items
""").show()

# Joinability with the dimension silver tables (info for the gold layer)
spark.sql("""
    SELECT
      COUNT(DISTINCT oi.customer_id) AS distinct_customers,
      COUNT(DISTINCT CASE WHEN c.customer_id IS NULL THEN oi.customer_id END) AS unmatched_customers,
      COUNT(DISTINCT oi.product_id) AS distinct_products,
      COUNT(DISTINCT CASE WHEN p.product_id IS NULL THEN oi.product_id END) AS unmatched_products
    FROM ecommerce.silver.silver_order_items oi
    LEFT JOIN ecommerce.silver.silver_customers c ON oi.customer_id = c.customer_id
    LEFT JOIN ecommerce.silver.silver_products p ON oi.product_id = p.product_id
""").show()

spark.sql("""
    SELECT discount_pct, COUNT(*) c
    FROM ecommerce.silver.silver_order_items
    GROUP BY discount_pct ORDER BY c DESC LIMIT 5
""").show()

display(spark.table("ecommerce.silver.silver_order_items").limit(10))